## Junção das bases de dados

Nesse notebook é feita a junção das bases de dados e a configuração final para a análise

In [1]:
import re
import unicodedata
import geopandas as gpd
from utils.load_csv import load_csv
from utils.load_shp import load_shp
from utils.save_shp import save_shp

In [2]:
votos_local = load_csv("votos_local.csv")
pbf_distrito = load_csv("pbf_distrito.csv")
distritos = load_shp("distritos_filtrado.shp")

esperadas = {"id_local", "nome_local", "bairro", "lat", "lon", "qt_secoes", "qt_eleitores",
             "vt_val_flavio", "vt_val_lula", "vt_branco", "vt_nulo",
             "vt_validos", "vt_total", "local_especial"}
faltando = esperadas - set(votos_local.columns)
assert not faltando, f"votos_local.csv desatualizado, faltam colunas: {sorted(faltando)}"

assert votos_local["id_local"].is_unique
assert votos_local[["lat", "lon"]].notna().all().all()
assert distritos.crs is not None
assert len(distritos) == 96, len(distritos)

votos_local["local_especial"] = votos_local["local_especial"].astype(bool)

def padronizar_nome(nome):
    nome = unicodedata.normalize("NFKD", str(nome).strip().lower())
    nome = "".join(c for c in nome if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", nome)

dist = distritos[["CD_DIST", "NM_DIST", "geometry"]]

pts = gpd.GeoDataFrame(
    votos_local,
    geometry=gpd.points_from_xy(votos_local["lon"], votos_local["lat"]),
    crs="EPSG:4326",
).to_crs(distritos.crs)

j = gpd.sjoin(pts, dist, how="left", predicate="within").drop(columns="index_right")

sem = j["CD_DIST"].isna()
if sem.any():
    print(f"{int(sem.sum())} locais sem distrito por 'within' (usando o mais próximo):")
    print(j.loc[sem, ["id_local", "nome_local", "bairro"]].to_string(index=False))
    prox = gpd.sjoin_nearest(
        pts.loc[sem].to_crs(31983),
        dist.to_crs(31983),
        how="left",
    ).drop(columns="index_right")
    prox = prox[~prox.index.duplicated(keep="first")]
    j.loc[sem, ["CD_DIST", "NM_DIST"]] = prox[["CD_DIST", "NM_DIST"]]

assert j["id_local"].is_unique, "Local duplicado no spatial join"
assert j["CD_DIST"].notna().all()

j["possivel_transito"] = j["id_local"].isin(["001-1562", "259-1104"])
j["remanejado"] = j["id_local"].isin(["256-1902", "320-1619"])

cols_soma = ["qt_secoes", "qt_eleitores", "vt_val_flavio", "vt_val_lula",
             "vt_branco", "vt_nulo", "vt_validos", "vt_total"]

def agrega(base_locais):
    d = (base_locais
         .groupby(["CD_DIST", "NM_DIST"], as_index=False)
         .agg(**{c: (c, "sum") for c in cols_soma},
              n_locais=("id_local", "nunique")))
    d["pct_val_lula"] = d["vt_val_lula"] / d["vt_validos"] * 100
    d["pct_val_flavio"] = d["vt_val_flavio"] / d["vt_validos"] * 100
    d["pct_comparecimento"] = d["vt_total"] / d["qt_eleitores"] * 100
    return d

df_dist = agrega(j[~j["local_especial"]])

df_dist_total = agrega(j)

assert df_dist["vt_validos"].gt(0).all()
assert df_dist_total["vt_total"].sum() == votos_local["vt_total"].sum()
assert len(df_dist) == 96, f"Distritos com votos na análise principal: {len(df_dist)}"

distritos["chave"] = distritos["NM_DIST"].map(padronizar_nome)
pbf = pbf_distrito.assign(chave=pbf_distrito["distrito"].map(padronizar_nome))

assert distritos["chave"].is_unique
assert pbf["chave"].is_unique

chk = distritos[["chave", "NM_DIST"]].merge(
    pbf[["chave", "distrito"]], on="chave", how="outer", indicator=True)

assert (chk["_merge"] == "both").all(), chk.loc[chk["_merge"] != "both"]

base = (
    distritos[["CD_DIST", "NM_DIST", "chave", "geometry"]]
    .merge(df_dist.drop(columns="NM_DIST"), on="CD_DIST", how="left", validate="one_to_one")
    .merge(pbf[["chave", "familias_pbf", "pop_total", "pbf_mil_hab"]],
           on="chave", how="left", validate="one_to_one")
    .drop(columns="chave")
    .rename(columns={"CD_DIST": "cd_dist", "NM_DIST": "distrito"})
)
base = gpd.GeoDataFrame(base, geometry="geometry", crs=distritos.crs)

base = base[[
    "cd_dist", "distrito", "geometry",
    "vt_val_lula", "vt_val_flavio", "vt_validos", "vt_branco", "vt_nulo", "vt_total",
    "pct_val_lula", "pct_val_flavio", "pct_comparecimento",
    "familias_pbf", "pop_total", "pbf_mil_hab",
    "n_locais", "qt_secoes", "qt_eleitores",
]].sort_values("distrito").reset_index(drop=True)

In [3]:
base.sample(5)

,cd_dist,distrito,geometry,vt_val_lula,vt_val_flavio,vt_validos,vt_branco,vt_nulo,vt_total,pct_val_lula,pct_val_flavio,pct_comparecimento,familias_pbf,pop_total,pbf_mil_hab,n_locais,qt_secoes,qt_eleitores
61,355030862,Pinheiros,"POLYGON ((-46.68209 -23.5579, -46.68208 -23.55...",32685,26364,59049,720,1382,61151,55.352335,44.647665,77.642204,2365,66007,35.829533,19,234,78760
90,355030890,Vila Mariana,"POLYGON ((-46.6404 -23.59965, -46.64075 -23.59...",57684,49080,106764,1356,2846,110966,54.029448,45.970552,79.406061,3812,127995,29.782413,24,394,139745
60,355030861,Perus,"POLYGON ((-46.7425 -23.38937, -46.74232 -23.38...",30062,21180,51242,836,1509,53587,58.666719,41.333281,79.663133,19315,87541,220.639472,15,188,67267
35,355030838,Jabaquara,"POLYGON ((-46.64639 -23.66874, -46.64659 -23.6...",73997,56603,130600,2002,3915,136517,56.659265,43.340735,79.283690,33454,212047,157.766910,38,485,172188
84,355030884,Vila Curuçá,"POLYGON ((-46.39773 -23.52413, -46.39771 -23.5...",47477,35704,83181,1460,3145,87786,57.076736,42.923264,80.869253,33423,137675,242.767387,24,304,108553


In [4]:
save_shp(base, "base.shp")

Shapefile salvo em data\base.shp


c:\Users\caiod\projetos\pbf_voto_sp\utils\save_shp.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(file_path)
c:\Users\caiod\projetos\pbf_voto_sp\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'vt_val_lula' to 'vt_val_lul'
  ogr_write(
c:\Users\caiod\projetos\pbf_voto_sp\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'vt_val_flavio' to 'vt_val_fla'
  ogr_write(
c:\Users\caiod\projetos\pbf_voto_sp\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'pct_val_lula' to 'pct_val_lu'
  ogr_write(
c:\Users\caiod\projetos\pbf_voto_sp\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'pct_val_flavio' to 'pct_val_fl'
  ogr_write(
c:\Users\caiod\projetos\pbf_voto_sp\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field 